<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 10. GPT-1 Generative Pre-Training — Improving Language Understanding by Generative Pre-Training

- **원 논문:** [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
- **저자 / 발표:** Alec Radford, Karthik Narasimhan, Tim Salimans, Ilya Sutskever · OpenAI Technical Report (2018)
- **난이도 / 예상 시간:** 중상급 · 약 85분
- **선수 지식:** Transformer decoder, causal LM loss, fine-tuning, 분류 cross entropy

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

작은 decoder-only Transformer를 causal language modeling으로 사전학습하고, 마지막 token 표현의 분류 head와 보조 LM 손실을 함께 미세조정한다. 미래 token 불변성과 사전학습/미세조정 metric을 통해 GPT-1의 두 단계 절차를 재현한다.

**원 논문과 다른 것**

GPT-1은 BooksCorpus(7천 권 이상)의 연속 텍스트로 12층, 768-hidden, 12-head, 약 1.17억 파라미터 Transformer를 사전학습한 뒤 여러 NLP benchmark에 미세조정했다. 여기서는 8개 고정 수열과 1층 24-hidden 모형을 사용한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.1, 식 (1): corpus token의 autoregressive likelihood L1 | `lm_inputs`/`lm_targets` shift와 causal LM cross entropy | LM loss 감소율을 assert하고 곡선으로 표시 |
| §3.1, 식 (2): multi-layer Transformer decoder h_l | `CausalSelfAttention`, `GPTBlock`, token+position embedding | logit shape와 미래 token에 대한 prefix 불변성 검사 |
| §3.1, 식 (2): P(u)=softmax(h_n W_e^T)의 embedding weight 재사용 | `TinyGPT.lm_logits`의 `F.linear(hidden, token_embedding.weight)` | LM vocabulary 축과 embedding parameter identity를 코드로 확인 |
| §3.2, 식 (3): 마지막 Transformer activation을 선형 분류기로 전달 | `classification_head(hidden[:, -1])` | 합성 분류 accuracy로 평가 |
| §3.2, 식 (4): L3 = L2 + λ·L1 보조 언어모델 목적 | `classification_loss + 0.2 * language_model_loss` | 두 loss가 유한한지와 최종 accuracy를 assert |
| §3.3 및 Figure 1: 구조화 입력을 ordered token sequence로 변환 | `sequences`가 BOS/content/EOS를 하나의 연속 token sequence로 구성 | 고정 길이와 경계 token을 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\mathcal{L}_{\mathrm{LM}}=-\sum_{t=1}^{T}
\log p(x_t\mid x_{<t}),\qquad
\mathcal{L}=\mathcal{L}_{\mathrm{task}}+\lambda\mathcal{L}_{\mathrm{LM}}
$$

- $\mathcal{L}_{\mathrm{LM}}$은 비라벨 문장의 다음 token 예측, $\mathcal{L}_{\mathrm{task}}$는 감독 목표입니다.
- 동일 Transformer decoder를 pretraining 후 작은 태스크에 fine-tuning하는 연결을 재현합니다.
- causal model, LM batch, task head, 두 loss를 따로 작성해 objective 경계를 보입니다.
- token logit은 $[B,T,V]$, task logit은 sequence representation으로부터 $[B,C]$가 됩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.1, 식 (1): corpus token의 autoregressive likelihood L1
- **노트북 구현:** `lm_inputs`/`lm_targets` shift와 causal LM cross entropy
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** LM loss 감소율을 assert하고 곡선으로 표시를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** GPT-1은 BooksCorpus(7천 권 이상)의 연속 텍스트로 12층, 768-hidden, 12-head, 약 1.17억 파라미터 Transformer를 사전학습한 뒤 여러 NLP benchmark에 미세조정했다. 여기서는 8개 고정 수열과 1층 24-hidden 모형을 사용한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

GPT-1의 절차는 (1) 레이블 없는 연속 텍스트에서 왼쪽 문맥만 보는 language model
likelihood $L_1$을 최대화하고, (2) 같은 Transformer를 task input에 맞게 미세조정하는
것이다. 지도 목적 $L_2$에 보조 LM 목적을 더한 $L_3=L_2+\lambda L_1$도 사용한다.
코드는 likelihood 최대화 대신 부호를 바꾼 cross-entropy를 최소화한다.

`nn.TransformerEncoder` 한 줄로 숨기지 않고, 식 (2)의 causal self-attention,
residual, LayerNorm, GELU feed-forward를 `CausalSelfAttention`과 `GPTBlock`으로
분리한다. GPT-1은 원 Transformer를 바탕으로 하므로 여기서는 residual 뒤에
LayerNorm을 두는 post-LN 순서를 쓴다. 언어모델 출력은 식 (2)처럼 token
embedding weight를 전치해 재사용한다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 1010
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())
BOS, EOS, VOCAB_SIZE = 1, 2, 16

### 구현 단계 2 · 논문 주장 확인 실험

- **원문 위치:** §3.1, 식 (1): corpus token의 autoregressive likelihood L1 / §3.3 및 Figure 1: 구조화 입력을 ordered token sequence로 변환
- **이 셀의 책임:** `lm_inputs`/`lm_targets` shift와 causal LM cross entropy / `sequences`가 BOS/content/EOS를 하나의 연속 token sequence로 구성
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** LM loss 감소율을 assert하고 곡선으로 표시 / 고정 길이와 경계 token을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
content = torch.tensor(
    [
        [3, 4, 5, 6],
        [3, 4, 5, 7],
        [3, 5, 4, 6],
        [3, 5, 4, 7],
        [8, 9, 10, 11],
        [8, 9, 10, 12],
        [8, 10, 9, 11],
        [8, 10, 9, 12],
    ]
)
labels = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
bos = torch.full((len(content), 1), BOS, dtype=torch.long)
eos = torch.full((len(content), 1), EOS, dtype=torch.long)
sequences = torch.cat((bos, content, eos), dim=1)
lm_inputs, lm_targets = sequences[:, :-1], sequences[:, 1:]
content, labels, sequences, lm_inputs, lm_targets = ACCELERATOR.move(
    content, labels, sequences, lm_inputs, lm_targets
)

assert sequences.shape == (8, 6)
assert torch.all(sequences[:, 0] == BOS) and torch.all(sequences[:, -1] == EOS)
assert torch.equal(lm_inputs[:, 1:], lm_targets[:, :-1])

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.1, 식 (2): multi-layer Transformer decoder h_l / §3.1, 식 (2): P(u)=softmax(h_n W_e^T)의 embedding weight 재사용
- **이 셀의 책임:** `CausalSelfAttention`, `GPTBlock`, token+position embedding / `TinyGPT.lm_logits`의 `F.linear(hidden, token_embedding.weight)`
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** logit shape와 미래 token에 대한 prefix 불변성 검사 / LM vocabulary 축과 embedding parameter identity를 코드로 확인. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_key_value = nn.Linear(
            d_model,
            3 * d_model,
            bias=False,
        )
        self.output_projection = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

    def _split_heads(self, tensor):
        batch, length, width = tensor.shape
        split = tensor.view(
            batch,
            length,
            self.num_heads,
            width // self.num_heads,
        )
        return split.transpose(1, 2)

    def _merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        merged = tensor.transpose(1, 2).contiguous()
        return merged.view(
            batch,
            length,
            self.num_heads * self.head_dim,
        )

    def forward(self, hidden):
        query, key, value = self.query_key_value(hidden).chunk(3, dim=-1)
        query = self._split_heads(query)
        key = self._split_heads(key)
        value = self._split_heads(value)
        scores = query @ key.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)
        length = hidden.shape[1]
        blocked = torch.ones(
            length,
            length,
            dtype=torch.bool,
            device=hidden.device,
        ).triu(diagonal=1)
        scores = scores.masked_fill(
            blocked.view(1, 1, length, length),
            torch.finfo(scores.dtype).min,
        )
        weights = torch.softmax(scores, dim=-1)
        context = self._merge_heads(weights @ value)
        return self.output_projection(context), weights


class GPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention = CausalSelfAttention(d_model, num_heads)
        self.attention_norm = nn.LayerNorm(d_model)
        self.feed_forward_input = nn.Linear(d_model, d_ff)
        self.feed_forward_output = nn.Linear(d_ff, d_model)
        self.feed_forward_norm = nn.LayerNorm(d_model)

    def forward(self, hidden):
        attention_output, weights = self.attention(hidden)
        hidden = self.attention_norm(hidden + attention_output)
        expanded = self.feed_forward_input(hidden)
        feed_forward_output = self.feed_forward_output(F.gelu(expanded))
        hidden = self.feed_forward_norm(hidden + feed_forward_output)
        return hidden, weights


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, max_length, d_model=24, num_heads=4):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.blocks = nn.ModuleList([GPTBlock(d_model, num_heads, d_ff=2 * d_model)])
        self.classification_head = nn.Linear(d_model, 2)

    def encode(self, token_ids):
        length = token_ids.shape[1]
        positions = torch.arange(
            length,
            device=token_ids.device,
        ).unsqueeze(0)
        hidden = self.token_embedding(token_ids)
        hidden = hidden + self.position_embedding(positions)
        for block in self.blocks:
            hidden, _ = block(hidden)
        return hidden

    def lm_logits(self, token_ids):
        hidden = self.encode(token_ids)
        return F.linear(hidden, self.token_embedding.weight)

    def classification_logits(self, token_ids):
        return self.classification_head(self.encode(token_ids)[:, -1])


torch.manual_seed(SEED)
model = TinyGPT(VOCAB_SIZE, max_length=sequences.shape[1]).to(DEVICE)
assert model.lm_logits(lm_inputs).shape == (*lm_inputs.shape, VOCAB_SIZE)
assert model.classification_logits(sequences).shape == (len(sequences), 2)
assert len(model.blocks) == 1

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3.1, 식 (1): corpus token의 autoregressive likelihood L1
- **이 셀의 책임:** `lm_inputs`/`lm_targets` shift와 causal LM cross entropy
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** LM loss 감소율을 assert하고 곡선으로 표시. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
pretrain_optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
pretrain_losses = []
for _ in range(70):
    model.train()
    pretrain_optimizer.zero_grad()
    logits = model.lm_logits(lm_inputs)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), lm_targets.reshape(-1))
    loss.backward()
    pretrain_optimizer.step()
    pretrain_losses.append(float(loss.detach()))

assert pretrain_losses[-1] < 0.45 * pretrain_losses[0]

model.eval()
original = lm_inputs[:1].clone()
changed = original.clone()
changed[:, 3:] = changed.new_tensor([[13, 14]])
with torch.inference_mode():
    original_logits = model.lm_logits(original)
    changed_logits = model.lm_logits(changed)
prefix_change = (original_logits[:, :3] - changed_logits[:, :3]).abs().max().item()
assert prefix_change < 1e-6
print(
    f"pretrain_loss={pretrain_losses[0]:.3f}->{pretrain_losses[-1]:.3f}, "
    f"prefix_change={prefix_change:.2e}"
)

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** §3.2, 식 (3): 마지막 Transformer activation을 선형 분류기로 전달 / §3.2, 식 (4): L3 = L2 + λ·L1 보조 언어모델 목적
- **이 셀의 책임:** `classification_head(hidden[:, -1])` / `classification_loss + 0.2 * language_model_loss`
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 합성 분류 accuracy로 평가 / 두 loss가 유한한지와 최종 accuracy를 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
finetune_optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
supervised_losses, auxiliary_lm_losses = [], []
for _ in range(60):
    model.train()
    finetune_optimizer.zero_grad()
    class_logits = model.classification_logits(sequences)
    lm_logits = model.lm_logits(lm_inputs)
    supervised_loss = F.cross_entropy(class_logits, labels)
    auxiliary_lm_loss = F.cross_entropy(
        lm_logits.reshape(-1, VOCAB_SIZE), lm_targets.reshape(-1)
    )
    joint_loss = supervised_loss + 0.2 * auxiliary_lm_loss
    joint_loss.backward()
    finetune_optimizer.step()
    supervised_losses.append(float(supervised_loss.detach()))
    auxiliary_lm_losses.append(float(auxiliary_lm_loss.detach()))

model.eval()
with torch.inference_mode():
    predictions = model.classification_logits(sequences).argmax(dim=-1)
classification_accuracy = (predictions == labels).float().mean().item()
assert classification_accuracy >= 0.875
assert math.isfinite(supervised_losses[-1])
assert math.isfinite(auxiliary_lm_losses[-1])
print(f"fine-tune accuracy={classification_accuracy:.1%}")

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1, 식 (1): corpus token의 autoregressive likelihood L1 / §3.2, 식 (3): 마지막 Transformer activation을 선형 분류기로 전달 / §3.2, 식 (4): L3 = L2 + λ·L1 보조 언어모델 목적
- **이 셀의 책임:** `lm_inputs`/`lm_targets` shift와 causal LM cross entropy / `classification_head(hidden[:, -1])` / `classification_loss + 0.2 * language_model_loss`
- **Tensor shape 계약:** tokens [B,T] → LM logits [B,T,V]와 task logits [B,C]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** LM loss 감소율을 assert하고 곡선으로 표시 / 합성 분류 accuracy로 평가 / 두 loss가 유한한지와 최종 accuracy를 assert. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
axes[0].plot(pretrain_losses, label="pretrain LM")
axes[0].plot(auxiliary_lm_losses, label="fine-tune auxiliary LM")
axes[0].set(title="generative objectives", xlabel="step", ylabel="cross entropy")
axes[0].legend()
axes[1].bar(
    ["error", "accuracy"],
    [1 - classification_accuracy, classification_accuracy],
)
axes[1].set(title="downstream classification", ylim=(0, 1.05))
plt.tight_layout()
plt.show()

## 해석 체크

보조 LM loss가 미세조정 중 어떤 regularizer 역할을 하는지, 그리고 이 실습의
`classification_logits`가 GPT-1 Figure 1의 task-aware input transform 전체를
재현하지 않는 이유를 적어 보세요.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `GPT-1은 BooksCorpus(7천 권 이상)의 연속 텍스트로 12층, 768-hidden, 12-head, 약 1.17억 파라미터 Transformer를 사전학습한 뒤 여러 NLP benchmark에 미세조정했다. 여기서는 8개 고정 수열과 1층 24-hidden 모형을 사용한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.